In [1]:
!pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
import sqlite3


Setup completed successfully!


### 1. Data Ingestion & Extraction Phase
In the **Extraction phase**, I ingested historical daily market data for 5 Saudi equities using the `yfinance` API into a centralized Pandas DataFrame.

In [6]:
saudi_equities = {
    '2222.SR': {'name': 'Saudi Aramco', 'sector': 'Energy'},
    '1120.SR': {'name': 'Al Rajhi Bank', 'sector': 'Banking'},
    '7010.SR': {'name': 'stc', 'sector': 'Telecom'},
    '2280.SR': {'name': 'Almarai', 'sector': 'Consumer Staples'},
    '1211.SR': {'name': 'Maaden', 'sector': 'Materials'}
}

all_data = []

for ticker, info in saudi_equities.items():
    print(f"Downloading data for: {ticker} ({info['name']})...")
    df_temp = yf.download(ticker, period="2y", interval="1d", auto_adjust=False)

    df_temp = df_temp.reset_index()

    if isinstance(df_temp.columns, pd.MultiIndex):
        df_temp.columns = [col[0] for col in df_temp.columns]

    df_temp['Ticker'] = ticker

    all_data.append(df_temp)

# دمج كل الأسهم في جدول واحد
df_raw = pd.concat(all_data, ignore_index=True)

print(f"Total records fetched: {len(df_raw)}")
display(df_raw.head())

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Total records fetched: 2505


,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
0,2024-11-04,24.693537,27.500000,27.549999,27.100000,27.150000,23073759,2222.SR
1,2024-11-05,24.738436,27.549999,27.549999,27.299999,27.450001,20287674,2222.SR
2,2024-11-06,24.738436,27.549999,27.549999,27.350000,27.549999,16718090,2222.SR
3,2024-11-07,25.142511,28.000000,28.100000,27.650000,27.650000,38332150,2222.SR
4,2024-11-10,25.187407,28.049999,28.200001,27.950001,28.049999,18968085,2222.SR


### 2. Data Cleaning & Quality Checks (Transformation Phase)
In this phase, I validated data integrity by casting columns to proper data types, eliminating composite-key duplicates (`Date`, `Ticker`), verifying non-null constraints, and applying domain-specific business rules (ensuring strictly positive prices and verifying that High >= Low).

In [12]:

df_clean = df_raw.copy()

# (Data Types)
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

initial_count = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Date', 'Ticker'])
duplicates_removed = initial_count - len(df_clean)

null_summary = df_clean[numeric_cols].isnull().sum()

df_clean = df_clean.dropna(subset=['Close', 'Volume'])

invalid_prices = df_clean[(df_clean['Close'] <= 0) | (df_clean['High'] < df_clean['Low'])]

print("Data Quality Report ")
print(f"Total Rows Before: {initial_count}")
print(f"Duplicates Removed: {duplicates_removed}")
print(f"Invalid Logic Rows: {len(invalid_prices)}")
print(f"Missing Values Per Column:\n{null_summary}")
print(f"\nFinal Cleaned Rows: {len(df_clean)}")

Data Quality Report 
Total Rows Before: 2505
Duplicates Removed: 0
Invalid Logic Rows: 0
Missing Values Per Column:
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Final Cleaned Rows: 2505


### 3. Data Storage & Relational Modeling (Load Phase)
In this phase, I implemented the loading process into a local SQLite database (`saudi_market.db`). I architected a normalized star-schema consisting of a dimension table (`dim_companies`) for stock metadata and a fact table (`fact_stock_prices`) for historical daily trading records, enforced via a Foreign Key constraint.

In [14]:


import sqlite3

db_path = "saudi_market.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_companies (
    ticker TEXT PRIMARY KEY,
    company_name TEXT NOT NULL,
    sector TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_stock_prices (
    record_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT NOT NULL,
    ticker TEXT NOT NULL,
    open REAL,
    high REAL,
    low REAL,
    close REAL,
    adj_close REAL,
    volume INTEGER,
    FOREIGN KEY (ticker) REFERENCES dim_companies (ticker),
    UNIQUE(date, ticker)
);
""")
conn.commit()

companies_rows = [
    (ticker, info['name'], info['sector'])
    for ticker, info in saudi_equities.items()
]

cursor.executemany("""
INSERT OR REPLACE INTO dim_companies (ticker, company_name, sector)
VALUES (?, ?, ?);
""", companies_rows)


df_to_load = df_clean.copy()
df_to_load['Date'] = df_to_load['Date'].dt.strftime('%Y-%m-%d')

price_records = df_to_load[['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].values.tolist()

cursor.executemany("""
INSERT OR IGNORE INTO fact_stock_prices (date, ticker, open, high, low, close, adj_close, volume)
VALUES (?, ?, ?, ?, ?, ?, ?, ?);
""", price_records)

conn.commit()

cursor.execute("SELECT COUNT(*) FROM dim_companies;")
companies_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM fact_stock_prices;")
prices_count = cursor.fetchone()[0]

print("Database Load Verification ")
print(f"Database file created: {db_path}")
print(f"Total Companies loaded (dim_companies): {companies_count}")
print(f"Total Daily Price records loaded (fact_stock_prices): {prices_count}")

conn.close()

Database Load Verification 
Database file created: saudi_market.db
Total Companies loaded (dim_companies): 5
Total Daily Price records loaded (fact_stock_prices): 2505


### 4. Financial KPI Analysis via SQL (Analytics Phase)
In this phase, I queried the SQLite relational database to extract core market performance metrics. Using SQL window functions, aggregations, and multi-table JOINs, I computed total cumulative returns, overall price ranges, average trading liquidity, and sector-level performance benchmarks.

In [17]:


import sqlite3
import pandas as pd

conn = sqlite3.connect("saudi_market.db")

query_summary = """
SELECT
    c.ticker,
    c.company_name,
    c.sector,
    ROUND(MIN(p.close), 2) AS min_close_price,
    ROUND(MAX(p.close), 2) AS max_close_price,
    ROUND(AVG(p.close), 2) AS avg_close_price,
    CAST(AVG(p.volume) AS INTEGER) AS avg_daily_volume
FROM fact_stock_prices p
JOIN dim_companies c ON p.ticker = c.ticker
GROUP BY c.ticker, c.company_name, c.sector
ORDER BY avg_daily_volume DESC;
"""

query_returns = """
WITH ranked_prices AS (
    SELECT
        ticker,
        date,
        close,
        ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date ASC) AS rn_first,
        ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date DESC) AS rn_last
    FROM fact_stock_prices
),
start_prices AS (
    SELECT ticker, close AS start_price, date AS start_date
    FROM ranked_prices
    WHERE rn_first = 1
),
end_prices AS (
    SELECT ticker, close AS end_price, date AS end_date
    FROM ranked_prices
    WHERE rn_last = 1
)
SELECT
    c.company_name,
    c.sector,
    s.ticker,
    s.start_date,
    ROUND(s.start_price, 2) AS start_price,
    e.end_date,
    ROUND(e.end_price, 2) AS end_price,
    ROUND(((e.end_price - s.start_price) / s.start_price) * 100, 2) AS total_return_pct
FROM start_prices s
JOIN end_prices e ON s.ticker = e.ticker
JOIN dim_companies c ON s.ticker = c.ticker
ORDER BY total_return_pct DESC;
"""

df_summary_kpi = pd.read_sql_query(query_summary, conn)
df_returns_kpi = pd.read_sql_query(query_returns, conn)

conn.close()

print("1. Stock Summary & Average Daily Volume ")
display(df_summary_kpi)

print("\n 2. Cumulative Total Returns (Ranked) ")
display(df_returns_kpi)

1. Stock Summary & Average Daily Volume 


,ticker,company_name,sector,min_close_price,max_close_price,avg_close_price,avg_daily_volume
0,2222.SR,Saudi Aramco,Energy,23.13,29.00,25.98,11842700
1,1120.SR,Al Rajhi Bank,Banking,58.10,73.33,65.47,5417486
2,7010.SR,stc,Telecom,39.65,48.10,43.09,3276287
3,1211.SR,Maaden,Materials,39.85,79.50,58.01,1926077
4,2280.SR,Almarai,Consumer Staples,39.86,61.10,48.51,827479



 2. Cumulative Total Returns (Ranked) 


,company_name,sector,ticker,start_date,start_price,end_date,end_price,total_return_pct
0,Maaden,Materials,1211.SR,2024-11-04,54.90,2026-11-04,57.90,5.46
1,Al Rajhi Bank,Banking,1120.SR,2024-11-04,59.07,2026-11-04,61.50,4.12
2,stc,Telecom,7010.SR,2024-11-04,42.30,2026-11-04,41.48,-1.94
3,Saudi Aramco,Energy,2222.SR,2024-11-04,27.50,2026-11-04,26.24,-4.58
4,Almarai,Consumer Staples,2280.SR,2024-11-04,55.70,2026-11-04,42.30,-24.06


### 5. Predictive Modeling: Machine Learning Baseline
In this section, I developed a supervised classification baseline using `scikit-learn` to predict next-day price direction (Up vs. Down). Technical features—such as simple moving averages (SMA), daily volatility spread, and volume momentum—were engineered chronologically. Data was partitioned using a strict time-series split to prevent lookahead bias.

In [19]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

conn = sqlite3.connect("saudi_market.db")
query = """
SELECT date, ticker, open, high, low, close, volume
FROM fact_stock_prices
ORDER BY ticker, date ASC;
"""
df_ml = pd.read_sql_query(query, conn)
conn.close()

#  (Feature Engineering)
feature_dfs = []
for ticker, group in df_ml.groupby('ticker'):
    g = group.copy().sort_values('date')

    g['Daily_Return'] = g['close'].pct_change()

    g['High_Low_Pct'] = (g['high'] - g['low']) / g['low']

    g['SMA_5'] = g['close'].rolling(window=5).mean()
    g['SMA_15'] = g['close'].rolling(window=15).mean()
    g['SMA_Ratio'] = g['SMA_5'] / g['SMA_15']

    g['Volume_Change'] = g['volume'].pct_change()

    g['Target'] = (g['close'].shift(-1) > g['close']).astype(int)

    feature_dfs.append(g)

df_features = pd.concat(feature_dfs, ignore_index=True)

df_features = df_features.replace([np.inf, -np.inf], np.nan)
df_features = df_features.dropna().reset_index(drop=True)

feature_cols = ['Daily_Return', 'High_Low_Pct', 'SMA_Ratio', 'Volume_Change']
X = df_features[feature_cols]
y = df_features['Target']

# (Time-Series Split: 80% تدريب، 20% اختبار)
split_index = int(len(df_features) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# (Random Forest)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("=== Machine Learning Baseline Evaluation ===")
print(f"Overall Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Down/Flat (0)', 'Up (1)']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

=== Machine Learning Baseline Evaluation ===
Overall Accuracy: 54.20%

Classification Report:
               precision    recall  f1-score   support

Down/Flat (0)       0.54      0.95      0.69       258
       Up (1)       0.50      0.06      0.11       218

     accuracy                           0.54       476
    macro avg       0.52      0.50      0.40       476
 weighted avg       0.52      0.54      0.43       476

Confusion Matrix:
[[244  14]
 [204  14]]


### 6. Model Evaluation & Performance Metrics
In this section, I summarized the Random Forest baseline results across core classification metrics (Accuracy, Precision, Recall, and F1-score) and mapped the test set outcomes into a structured confusion matrix to evaluate class imbalance and directional bias.


In [20]:


import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# (Classification Report)
report_dict = classification_report(y_test, y_pred, target_names=['Down / Flat (0)', 'Up (1)'], output_dict=True)
df_metrics = pd.DataFrame(report_dict).transpose().round(2)

# تنسيق  الجدول
df_metrics = df_metrics.rename(columns={
    'precision': 'Precision',
    'recall': 'Recall',
    'f1-score': 'F1-Score',
    'support': 'Test Samples'
})
df_metrics['Test Samples'] = df_metrics['Test Samples'].astype(int)

# (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(
    cm,
    index=['Actual: Down/Flat (0)', 'Actual: Up (1)'],
    columns=['Predicted: Down/Flat (0)', 'Predicted: Up (1)']
)

# عرض النتائج
print(f"Overall Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("--- 1. Classification Performance Metrics ---")
display(df_metrics)

print("\n--- 2. Confusion Matrix ---")
display(df_cm)

Overall Model Accuracy: 54.20%

--- 1. Classification Performance Metrics ---


,Precision,Recall,F1-Score,Test Samples
Down / Flat (0),0.54,0.95,0.69,258
Up (1),0.50,0.06,0.11,218
accuracy,0.54,0.54,0.54,0
macro avg,0.52,0.50,0.40,476
weighted avg,0.52,0.54,0.43,476



--- 2. Confusion Matrix ---


,Predicted: Down/Flat (0),Predicted: Up (1)
Actual: Down/Flat (0),244,14
Actual: Up (1),204,14


In [23]:

from google.colab import files
df_clean.to_csv('saudi_equities.csv', index=False)
files.download('saudi_equities.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>